In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KDTree

df = pd.read_csv(
    "results/galaxy_coordinates.csv.gz",
    dtype={"specObjID": "string"},
)

xyz = df[["X", "Y", "Z"]].to_numpy()

df.shape

(494502, 8)

In [ ]:
K_VALUES = [5, 10, 20]
BATCH_SIZE = 20000

if len(df) <= max(K_VALUES):
    raise ValueError("은하 수 부족")

tree = KDTree(xyz)
neighbor_distances = np.empty((len(df), len(K_VALUES)))

for start in range(0, len(df), BATCH_SIZE):
    end = min(start + BATCH_SIZE, len(df))

    distances, _ = tree.query(
        xyz[start:end],
        k=max(K_VALUES) + 1,
    )

    neighbor_distances[start:end] = distances[:, K_VALUES]

In [ ]:
environment = df[["specObjID", "z"]].copy()

for column, k in enumerate(K_VALUES):
    environment[f"r{k}"] = neighbor_distances[:, column]

environment.to_csv(
    "results/galaxy_environment.csv.gz",
    index=False,
    compression="gzip",
)

environment[["r5", "r10", "r20"]].describe()

In [ ]:
environment = pd.read_csv(
    "results/galaxy_environment.csv.gz",
    dtype={"specObjID": "string"},
)

sample = environment.sample(
    n=min(50000, len(environment)),
    random_state=42,
)

plt.figure(figsize=(8, 5))
plt.scatter(
    sample["z"],
    sample["r10"],
    s=1,
    alpha=0.2,
)
plt.xlabel("Redshift")
plt.ylabel(r"10th-neighbor distance [$h^{-1}\mathrm{Mpc}$]")
plt.tight_layout()
plt.savefig("results/neighbor_distance_vs_redshift.png", dpi=180)
plt.show()